# GWAS Catalog

The [GWAS Catalog](https://www.ebi.ac.uk/gwas/) is a curated, publicly available repository of all published genome-wide association studies (GWAS). Maintained by EMBL-EBI and NHGRI, it captures SNP-trait associations from thousands of studies, providing a central resource for understanding the genetic basis of complex traits and diseases.

## Key Data Types

| Data Type | Description | Example |
|-----------|-------------|---------|
| Studies | Metadata for each published GWAS | Accession ID, sample sizes, platform, ancestry |
| Associations | SNP–trait associations with effect sizes | rs ID, p-value, odds ratio, beta, risk allele |
| SNPs | Single nucleotide polymorphism details | Chromosomal position, functional context, mapped genes |
| EFO Traits | Ontology-mapped trait terms (Experimental Factor Ontology) | `EFO_0000400` → diabetes mellitus |

## Reference

- REST API base URL: `https://www.ebi.ac.uk/gwas/rest/api`
- API documentation: https://www.ebi.ac.uk/gwas/rest/api
- GWAS Catalog portal: https://www.ebi.ac.uk/gwas/
- ELIXIR Core Data Resource: https://elixir-europe.org/platforms/data/core-data-resources

In [ ]:
import requests
import time
from pathlib import Path

import polars as pl

## TODO

- [x] Ingest data
  - [x] Connect to GWAS Catalog REST API and confirm access
  - [x] Fetch all studies (paginated) into a Polars DataFrame
  - [x] Fetch association data (SNP-trait associations) for a study of interest
  - [x] Parse and save data to `data/` directory with caching
- [ ] Explore and clean data
  - [ ] Inspect schema, check for nulls and duplicates
  - [ ] Normalise trait labels and ancestry fields
  - [ ] Filter to genome-wide significant associations (p < 5e-8)
- [ ] Analysis
  - [ ] Manhattan / QQ plot prerequisites
  - [ ] Trait enrichment analysis
  - [ ] Ancestry breakdown across studies
- [ ] Visualization
  - [ ] Manhattan plot of association p-values
  - [ ] Trait-by-ancestry heatmap
  - [ ] Publication trend over time
- [ ] Statistical analysis
  - [ ] Multiple hypothesis correction (Bonferroni, FDR/BH)
  - [ ] Effect size distribution analysis
  - [ ] Power analysis primer

## 1. Ingest Data

### 1.1 Connect to the GWAS Catalog REST API

In [ ]:
BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

# Standard headers: request JSON, identify the client
HEADERS = {
    "Accept": "application/json",
    "User-Agent": "elixir-of-life-demo/1.0 (https://github.com/elixir-of-life)",
}


def check_api_access(base_url: str = BASE_URL) -> dict:
    """Probe the GWAS Catalog REST API root and return the available resource links.

    Parameters
    ----------
    base_url : str
        Root URL of the GWAS Catalog REST API.

    Returns
    -------
    dict
        Parsed JSON response containing ``_links`` to all top-level resources.

    Raises
    ------
    requests.HTTPError
        If the server returns a non-2xx status code.
    """
    response = requests.get(base_url, headers=HEADERS, timeout=30)
    response.raise_for_status()  # raise on 4xx / 5xx
    return response.json()


api_root = check_api_access()

# Print the top-level resource names discovered from the API
print("GWAS Catalog REST API — available resources:")
for name in api_root.get("_links", {}).keys():
    print(f"  • {name}")

### 1.2 Fetch All Studies (Paginated)

In [ ]:
def fetch_studies_page(page: int, page_size: int = 500) -> dict:
    """Fetch a single page of studies from the GWAS Catalog REST API.

    Parameters
    ----------
    page : int
        Zero-indexed page number to retrieve.
    page_size : int, optional
        Number of records per page (default 500, max 1000).

    Returns
    -------
    dict
        Raw API response with ``_embedded.studies``, ``_links``, and ``page`` keys.

    Raises
    ------
    requests.HTTPError
        If the server returns a non-2xx status code.
    """
    params = {"page": page, "size": page_size}
    response = requests.get(
        f"{BASE_URL}/studies", headers=HEADERS, params=params, timeout=60
    )
    response.raise_for_status()
    return response.json()


def parse_study(study: dict) -> dict:
    """Flatten a raw study JSON object into a plain dictionary suitable for tabular storage.

    Parameters
    ----------
    study : dict
        A single study object from the ``_embedded.studies`` list.

    Returns
    -------
    dict
        Flat dictionary with selected fields extracted from nested structures.
    """
    pub = study.get("publicationInfo") or {}
    trait = study.get("diseaseTrait") or {}

    return {
        "accession_id": study.get("accessionId"),
        "pubmed_id": pub.get("pubmedId"),
        "title": pub.get("title"),
        "author": pub.get("author"),
        "publication_date": pub.get("publicationDate"),
        "journal": pub.get("journal"),
        "disease_trait": trait.get("trait"),
        "initial_sample_size": study.get("initialSampleSize"),
        "replication_sample_size": study.get("replicationSampleSize"),
        "snp_count": study.get("snpCount"),
        "imputed": study.get("imputed"),
        "full_pvalue_set": study.get("fullPvalueSet"),
    }


def fetch_all_studies(page_size: int = 500, max_pages: int | None = None) -> pl.DataFrame:
    """Retrieve all GWAS Catalog studies via paginated API calls.

    The API uses zero-indexed Spring Data pagination. We read ``page.totalPages``
    from the first response to know how many requests to make, then iterate with
    a short polite delay between requests.

    Parameters
    ----------
    page_size : int, optional
        Records per API request (default 500).
    max_pages : int or None, optional
        Cap the number of pages fetched (useful for testing). ``None`` means fetch all.

    Returns
    -------
    pl.DataFrame
        Polars DataFrame with one row per study and flattened columns.
    """
    # --- first page: discover total page count ---
    first = fetch_studies_page(page=0, page_size=page_size)
    page_meta = first["page"]
    total_pages = page_meta["totalPages"]
    total_elements = page_meta["totalElements"]

    if max_pages is not None:
        total_pages = min(total_pages, max_pages)

    print(f"Total studies: {total_elements:,} across {page_meta['totalPages']:,} pages")
    print(f"Fetching {total_pages} page(s) of {page_size} …")

    records = [parse_study(s) for s in first["_embedded"]["studies"]]

    # --- subsequent pages ---
    for p in range(1, total_pages):
        time.sleep(0.1)  # be polite to the API server
        data = fetch_studies_page(page=p, page_size=page_size)
        records.extend(parse_study(s) for s in data["_embedded"]["studies"])
        if (p + 1) % 10 == 0 or p == total_pages - 1:
            print(f"  … fetched page {p + 1}/{total_pages} ({len(records):,} records so far)")

    return pl.DataFrame(records)


# Fetch studies — cap to 10 pages (~5 000 studies) for a fast demo run;
# remove max_pages to ingest the full ~220k catalogue.
studies_df = fetch_all_studies(page_size=500, max_pages=10)

print(f"\nLoaded {len(studies_df):,} studies")
print(studies_df.schema)

### 1.3 Fetch Association Data for a Study of Interest

In [ ]:
# We use GCST000854 — a well-known BMI GWAS (Speliotes et al. 2010, Nature Genetics)
# with a tractable number of associations, making it ideal as a demo study.
STUDY_ACCESSION = "GCST000854"


def fetch_study_associations(
    accession_id: str, page_size: int = 500
) -> pl.DataFrame:
    """Fetch all SNP-trait associations for a given GWAS Catalog study.

    Iterates over all pages of the study's ``/associations`` sub-resource,
    flattening each association record into a tabular row.

    Parameters
    ----------
    accession_id : str
        GWAS Catalog study accession ID, e.g. ``"GCST000854"``.
    page_size : int, optional
        Records per API page (default 500).

    Returns
    -------
    pl.DataFrame
        One row per association with columns for SNP, locus, p-value,
        effect size, and risk allele information.

    Raises
    ------
    requests.HTTPError
        If any API request fails.
    """
    url = f"{BASE_URL}/studies/{accession_id}/associations"
    records = []
    page = 0

    while True:
        resp = requests.get(
            url, headers=HEADERS, params={"page": page, "size": page_size}, timeout=60
        )
        resp.raise_for_status()
        data = resp.json()

        embedded = data.get("_embedded", {})
        associations = embedded.get("associations", [])

        for assoc in associations:
            # Pull the first locus (most associations have exactly one)
            loci = assoc.get("loci") or []
            locus = loci[0] if loci else {}

            # Strongest risk allele info lives inside the locus
            risk_alleles = locus.get("strongestRiskAlleles") or []
            risk_allele = risk_alleles[0] if risk_alleles else {}

            # Author-reported genes at this locus (may be empty for intergenic hits)
            author_genes = locus.get("authorReportedGenes") or []
            gene_names = [g.get("geneName") for g in author_genes if g.get("geneName")]

            records.append({
                "study_accession": accession_id,
                "snp_id": risk_allele.get("riskAlleleName", "").split("-")[0],
                "risk_allele": risk_allele.get("riskAlleleName"),
                "risk_frequency": assoc.get("riskFrequency"),
                "pvalue": assoc.get("pvalue"),
                "pvalue_mlog": assoc.get("pvalueMlog"),        # -log10(p), precomputed
                "pvalue_description": assoc.get("pvalueDescription"),
                "or_per_copy": assoc.get("orPerCopyNum"),      # odds ratio
                "beta_num": assoc.get("betaNum"),               # beta coefficient
                "beta_unit": assoc.get("betaUnit"),
                "beta_direction": assoc.get("betaDirection"),
                "standard_error": assoc.get("standardError"),
                "range": assoc.get("range"),
                "snp_type": assoc.get("snpType"),               # novel / known
                "locus_description": locus.get("description"),
                "genes": ", ".join(gene_names) if gene_names else None,
            })

        page_meta = data.get("page", {})
        total_pages = page_meta.get("totalPages", 1)

        print(
            f"  Page {page + 1}/{total_pages} — {len(records)} associations fetched"
        )

        # Advance or stop when we have reached the final page
        if page + 1 >= total_pages:
            break
        page += 1
        time.sleep(0.1)  # polite delay

    return pl.DataFrame(records)


print(f"Fetching associations for study {STUDY_ACCESSION} …")
associations_df = fetch_study_associations(STUDY_ACCESSION)

print(f"\nLoaded {len(associations_df):,} associations")
associations_df.head(5)